In [ ]:
"""
Reference Area Creation for New Housing Development Areas (NHDA)

For each NHDA cluster, finds a matching reference area (RA) within ATKIS
settlement polygons (classes 41001 / 41006) nearby. Reference areas match
the NHDA in size (±20%) and must have a new_vs_existing buildings area
ratio of ≤0.1 (i.e. ≥90% of building footprint area is from existing
buildings).

After selection, each RA is adjusted along building footprint boundaries:
- RA smaller than NHDA → expand to fully include all buildings cut by the RA edge
- RA larger  than NHDA → shrink to fully exclude all buildings cut by the RA edge

Output: GeoPackage with reference areas only, with column `ra_id`
        copied from the source `nhda_id`.

Author: Agnes Zwick
Date: February 2026
"""

import geopandas as gpd
from shapely.geometry import Point
from shapely.ops import unary_union
from pathlib import Path
import numpy as np
import pandas as pd
import math
import sys
from contextlib import contextmanager
from datetime import datetime


# =============================================================================
# LOGGING
# =============================================================================

@contextmanager
def log_stdout_to_file(log_file):
    class TeeOutput:
        def __init__(self, path):
            self.terminal = sys.stdout
            self.log = open(path, "w", encoding="utf-8")
        def write(self, msg):
            self.terminal.write(msg)
            self.log.write(msg)
            self.log.flush()
        def flush(self):
            self.terminal.flush()
            self.log.flush()
        def close(self):
            self.log.close()

    tee = TeeOutput(log_file)
    old_stdout = sys.stdout
    sys.stdout = tee
    try:
        yield
    finally:
        sys.stdout = old_stdout
        tee.close()


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def chunked_unary_union(geometries, chunk_size=500):
    geom_list = list(geometries)
    n = len(geom_list)
    if n <= chunk_size:
        return unary_union(geom_list)
    chunks = []
    for i in range(0, n, chunk_size):
        chunk = geom_list[i:i + chunk_size]
        print(f"    Chunk {i//chunk_size + 1}/{(n-1)//chunk_size + 1} ({len(chunk)} polygons)...")
        chunks.append(unary_union(chunk))
    print(f"  Merging {len(chunks)} chunks...")
    return unary_union(chunks)


def create_grid_points(polygon, spacing=100):
    minx, miny, maxx, maxy = polygon.bounds
    points = []
    for x in np.arange(minx, maxx, spacing):
        for y in np.arange(miny, maxy, spacing):
            p = Point(x, y)
            if polygon.contains(p):
                points.append(p)
    return points


def create_circle_from_area(center_point, target_area):
    radius = math.sqrt(target_area / np.pi)
    return center_point.buffer(radius)


def calculate_new_vs_existing_ratio(geometry,
                                    new_gdf, new_sindex,
                                    existing_gdf, existing_sindex,
                                    debug=False):
    """
    Returns new_area / existing_area for building footprints within geometry.

    - Returns 0.0  if there are no buildings at all (no new, no existing)
      → treated as "all existing", ratio = 0, passes the filter.
    - Returns inf  if existing_area == 0 but new_area > 0
      → will fail the ≤0.1 filter as expected.
    """
    try:
        def clipped_area(gdf, sindex):
            hits = list(sindex.intersection(geometry.bounds))
            if not hits:
                return 0.0
            candidates = gdf.iloc[hits]
            clipped = candidates.geometry.intersection(geometry)
            return float(clipped[~clipped.is_empty].area.sum())

        new_area      = clipped_area(new_gdf,      new_sindex)
        existing_area = clipped_area(existing_gdf, existing_sindex)

        if new_area == 0.0:
            return 0.0             # no new buildings → ratio = 0
        if existing_area == 0.0:
            return float("inf")    # only new buildings → ratio = ∞
        return new_area / existing_area

    except Exception as e:
        if debug:
            print(f"      ERROR new_vs_existing ratio: {e}")
        return float("inf")        # fail safe: reject candidate


def find_valid_circles(grid_points, atkis_polygon, target_area,
                       new_gdf, new_sindex,
                       existing_gdf, existing_sindex,
                       tolerance=0.2, ratio_max=0.1, debug=False):
    valid_circles = []
    failed_area = failed_ratio = 0

    for point in grid_points:
        circle  = create_circle_from_area(point, target_area)
        clipped = circle.intersection(atkis_polygon)

        if clipped.is_empty:
            continue

        area_diff = abs(clipped.area - target_area) / target_area
        if area_diff > tolerance:
            failed_area += 1
            continue

        ratio = calculate_new_vs_existing_ratio(
            clipped, new_gdf, new_sindex, existing_gdf, existing_sindex
        )
        if ratio > ratio_max:
            failed_ratio += 1
            continue

        valid_circles.append({
            "geometry":        clipped,
            "center":          point,
            "area":            clipped.area,
            "area_diff_ratio": area_diff,
            "new_vs_existing": ratio
        })

    if debug and not valid_circles:
        print(f"      Tested {len(grid_points)} pts | "
              f"failed area: {failed_area} | failed ratio: {failed_ratio}")

    return valid_circles


# ← NEW -----------------------------------------------------------------------
def adjust_ra_to_buildings(ra_geom, nhda_area, all_bldg_gdf, all_bldg_sindex,
                            debug=False):
    """
    Snaps the RA boundary to building footprints (from LoD2_2025.gpkg).

    A building is considered "cut" when it intersects the RA but is not
    fully contained within it — i.e. the RA boundary runs through it.

    - RA area < NHDA area  →  expand: union RA with every cut building so
                               no building is partially inside/outside.
    - RA area ≥ NHDA area  →  shrink: subtract every cut building from the RA
                               so the boundary no longer bisects any footprint.

    Returns the (possibly unchanged) adjusted geometry.
    Falls back to the original geometry if the result would be empty or invalid.
    """
    ra_area = ra_geom.area

    # Candidate buildings whose bounding box overlaps the RA
    hits = list(all_bldg_sindex.intersection(ra_geom.bounds))
    if not hits:
        return ra_geom

    candidates = all_bldg_gdf.iloc[hits]

    # "Cut" buildings: intersect the RA but are NOT fully inside it
    cut_geoms = []
    for geom in candidates.geometry:
        if geom is None or geom.is_empty or not geom.is_valid:
            continue
        if ra_geom.intersects(geom) and not ra_geom.contains(geom):
            cut_geoms.append(geom)

    if not cut_geoms:
        if debug:
            print(f"      Building adjustment: no cut buildings found")
        return ra_geom

    cut_union = unary_union(cut_geoms)

    if ra_area < nhda_area:
        # Expand RA to swallow all cut buildings
        adjusted = ra_geom.union(cut_union)
        direction = "expanded"
    else:
        # Shrink RA to exclude all cut buildings
        adjusted = ra_geom.difference(cut_union)
        direction = "shrunk"

    if adjusted.is_empty or not adjusted.is_valid:
        if debug:
            print(f"      Building adjustment produced empty/invalid geometry — keeping original")
        return ra_geom

    area_change = adjusted.area - ra_area
    if debug:
        print(f"      Building adjustment ({direction}): "
              f"{ra_area:.0f} m² → {adjusted.area:.0f} m²  "
              f"(Δ {area_change:+.0f} m², {len(cut_geoms)} cut buildings)")

    return adjusted
# ← END NEW -------------------------------------------------------------------


# =============================================================================
# MAIN
# =============================================================================

def main():

    # -------------------------------------------------------------------------
    # CONFIGURATION
    # -------------------------------------------------------------------------

    NHDA_GPKG       = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\NHDA_residential_wsf2015_max10pct.gpkg"
    ATKIS_GPKG      = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\ATKIS\ATKIS_41001_41006.gpkg"
    NEW_BLDG_GPKG   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings.gpkg"
    EXIST_BLDG_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_existing.gpkg"
    ALL_BLDG_GPKG   = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025.gpkg"  # ← NEW
    OUTPUT_DIR      = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\Reference_Areas")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ATKIS object type codes to use as search space for RAs.
    # 41001 = Wohnbaufläche, 41006 = Fläche gemischter Nutzung
    # Set to None to skip column filtering (use all polygons in the file).
    ATKIS_CLASSES       = [41001, 41006]   # or None
    ATKIS_CODE_COL      = "OBJART"         # column holding the object type code

    BUFFER_DISTANCES    = [100, 200, 300, 400, 500, 1000, 1500, 2000]
    AREA_TOLERANCE      = 0.2   # ±20%
    GRID_SPACING        = 50    # metres
    NEW_VS_EXISTING_MAX = 0.1   # new_area / existing_area ≤ 10%
    DEBUG               = True

    LOG_FILE    = OUTPUT_DIR / f"log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    OUTPUT_GPKG = OUTPUT_DIR / "Reference_Areas.gpkg"

    with log_stdout_to_file(LOG_FILE):
        print("=" * 80)
        print("REFERENCE AREA CREATION")
        print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        print("=" * 80)

        # ---------------------------------------------------------------------
        # LOAD DATA
        # ---------------------------------------------------------------------

        print("\nLoading data...")
        nhda_gdf     = gpd.read_file(NHDA_GPKG)
        atkis_gdf    = gpd.read_file(ATKIS_GPKG)
        new_gdf      = gpd.read_file(NEW_BLDG_GPKG)
        exist_gdf    = gpd.read_file(EXIST_BLDG_GPKG)
        all_bldg_gdf = gpd.read_file(ALL_BLDG_GPKG)  # ← NEW
        print(f"  ✓ NHDA:             {len(nhda_gdf):,} clusters")
        print(f"  ✓ ATKIS:            {len(atkis_gdf):,} polygons  |  columns: {list(atkis_gdf.columns)}")
        print(f"  ✓ New buildings:    {len(new_gdf):,} footprints")
        print(f"  ✓ Existing bldgs:   {len(exist_gdf):,} footprints")
        print(f"  ✓ All buildings:    {len(all_bldg_gdf):,} footprints")  # ← NEW

        # Optional: filter ATKIS to relevant object type codes
        if ATKIS_CLASSES is not None:
            if ATKIS_CODE_COL not in atkis_gdf.columns:
                raise ValueError(
                    f"Column '{ATKIS_CODE_COL}' not found in ATKIS layer. "
                    f"Available columns: {list(atkis_gdf.columns)}"
                )
            atkis_gdf[ATKIS_CODE_COL] = pd.to_numeric(
                atkis_gdf[ATKIS_CODE_COL], errors="coerce"
            )
            atkis_gdf = atkis_gdf[atkis_gdf[ATKIS_CODE_COL].isin(ATKIS_CLASSES)]
            print(f"  ✓ ATKIS filtered to classes {ATKIS_CLASSES}: {len(atkis_gdf):,} polygons")

        # Align all layers to NHDA CRS
        target_crs = nhda_gdf.crs
        if target_crs.is_geographic:
            raise ValueError("CRS must be projected!")

        for name, gdf in [("ATKIS", atkis_gdf), ("new buildings", new_gdf),
                          ("existing buildings", exist_gdf), ("all buildings", all_bldg_gdf)]:  # ← NEW
            if gdf.crs != target_crs:
                print(f"  Reprojecting {name} → {target_crs}...")

        atkis_gdf    = atkis_gdf.to_crs(target_crs)    if atkis_gdf.crs    != target_crs else atkis_gdf
        new_gdf      = new_gdf.to_crs(target_crs)      if new_gdf.crs      != target_crs else new_gdf
        exist_gdf    = exist_gdf.to_crs(target_crs)    if exist_gdf.crs    != target_crs else exist_gdf
        all_bldg_gdf = all_bldg_gdf.to_crs(target_crs) if all_bldg_gdf.crs != target_crs else all_bldg_gdf  # ← NEW

        print(f"  ✓ CRS: {target_crs}")

        # Spatial indices
        print("\nBuilding spatial indices...")
        new_sindex      = new_gdf.sindex
        exist_sindex    = exist_gdf.sindex
        all_bldg_sindex = all_bldg_gdf.sindex  # ← NEW
        print("  ✓ Spatial indices ready")

        # ATKIS union (search space for RAs)
        print("\nCreating ATKIS union...")
        atkis_union = chunked_unary_union(atkis_gdf.geometry, chunk_size=500)
        print("  ✓ ATKIS union complete")

        # ---------------------------------------------------------------------
        # PROCESS EACH NHDA
        # ---------------------------------------------------------------------

        ra_rows      = []
        failed       = []
        meta_records = []

        print(f"\n{'='*80}")
        print(f"PROCESSING {len(nhda_gdf)} NHDA CLUSTERS")
        print(f"{'='*80}")

        for i, (_, row) in enumerate(nhda_gdf.iterrows()):
            cluster_id    = row["nhda_id"]
            nhda_geom     = row.geometry
            nhda_area     = nhda_geom.area
            nhda_centroid = nhda_geom.centroid

            print(f"\n[{i+1}/{len(nhda_gdf)}] {cluster_id}  |  area: {nhda_area:.0f} m²")

            if DEBUG:
                nhda_ratio = calculate_new_vs_existing_ratio(
                    nhda_geom, new_gdf, new_sindex, exist_gdf, exist_sindex
                )
                print(f"  NHDA new_vs_existing ratio: {nhda_ratio:.3f}")

            found = False

            for buf in BUFFER_DISTANCES:
                if found:
                    break
                print(f"  Buffer {buf}m...")

                search_area = nhda_geom.buffer(buf).intersection(atkis_union).difference(nhda_geom)
                if search_area.is_empty:
                    print(f"    No ATKIS area in buffer")
                    continue

                grid_pts = create_grid_points(search_area, spacing=GRID_SPACING)
                if not grid_pts:
                    print(f"    No grid points")
                    continue
                print(f"    {len(grid_pts)} grid points")

                circles = find_valid_circles(
                    grid_pts, search_area, nhda_area,
                    new_gdf, new_sindex,
                    exist_gdf, exist_sindex,
                    tolerance=AREA_TOLERANCE,
                    ratio_max=NEW_VS_EXISTING_MAX,
                    debug=DEBUG
                )

                if not circles:
                    print(f"    No valid circles")
                    continue

                best = min(circles, key=lambda c: c["center"].distance(nhda_centroid))
                dist = best["center"].distance(nhda_centroid)

                print(f"    ✓ RA found | area diff: {best['area_diff_ratio']*100:.1f}% "
                      f"| new_vs_existing: {best['new_vs_existing']:.3f} | dist: {dist:.0f}m")

                # ← NEW: adjust RA boundary to align with building footprints
                adjusted_geom = adjust_ra_to_buildings(
                    best["geometry"], nhda_area,
                    all_bldg_gdf, all_bldg_sindex,
                    debug=DEBUG
                )

                ra_attr = row.to_dict()
                ra_attr["geometry"]     = adjusted_geom          # ← NEW (was best["geometry"])
                ra_attr["cluster_type"] = "RA"
                ra_rows.append(ra_attr)

                meta_records.append({
                    "cluster_id":          cluster_id,
                    "buffer_m":            buf,
                    "ra_area_m2":          adjusted_geom.area,   # ← NEW (was best["area"])
                    "ra_area_raw_m2":      best["area"],          # ← NEW: pre-adjustment area
                    "nhda_area_m2":        nhda_area,
                    "area_diff_pct":       best["area_diff_ratio"] * 100,
                    "new_vs_existing":     best["new_vs_existing"],
                    "dist_to_nhda_m":      dist,
                    "n_valid_circles":     len(circles),
                    "n_grid_points":       len(grid_pts)
                })
                found = True

            if not found:
                print(f"  ✗ No RA found")
                failed.append(cluster_id)

        # ---------------------------------------------------------------------
        # BUILD OUTPUT (RA ONLY)
        # ---------------------------------------------------------------------

        print(f"\n{'='*80}")
        print("SAVING OUTPUT")
        print(f"{'='*80}")

        if ra_rows:
            ra_gdf = gpd.GeoDataFrame(ra_rows, crs=nhda_gdf.crs)
            if "nhda_id" not in ra_gdf.columns:
                raise ValueError("Column 'nhda_id' not found in RA output rows.")
            ra_gdf["ra_id"] = ra_gdf["nhda_id"]
        else:
            ra_gdf = gpd.GeoDataFrame(columns=["ra_id", "nhda_id", "geometry"], crs=nhda_gdf.crs)

        ra_gdf.to_file(OUTPUT_GPKG, driver="GPKG")
        print(f"✓ RA-only GPKG: {OUTPUT_GPKG.name}")
        print(f"  RA rows:  {len(ra_gdf)}")

        if meta_records:
            pd.DataFrame(meta_records).to_csv(
                OUTPUT_DIR / "reference_area_metadata.csv", index=False
            )
            print(f"✓ Metadata CSV saved")

        if failed:
            pd.DataFrame({"cluster_id": failed}).to_csv(
                OUTPUT_DIR / "failed_clusters.csv", index=False
            )

        # ---------------------------------------------------------------------
        # SUMMARY
        # ---------------------------------------------------------------------

        print(f"\n{'='*80}")
        print("SUMMARY")
        print(f"{'='*80}")
        print(f"  Total NHDA:  {len(nhda_gdf)}")
        print(f"  RA found:    {len(ra_rows)}  ({len(ra_rows)/len(nhda_gdf)*100:.1f}%)")
        print(f"  Failed:      {len(failed)}")

        if meta_records:
            meta_df = pd.DataFrame(meta_records)
            print(f"\n  Buffer distribution:")
            for buf, cnt in meta_df["buffer_m"].value_counts().sort_index().items():
                print(f"    {buf}m: {cnt}  ({cnt/len(meta_df)*100:.1f}%)")
            print(f"\n  Avg area deviation:    {meta_df['area_diff_pct'].mean():.2f}%")
            print(f"  Avg new_vs_existing:   {meta_df['new_vs_existing'].mean():.3f}")
            print(f"  Avg dist to NHDA:      {meta_df['dist_to_nhda_m'].mean():.0f} m")

        if failed:
            print(f"\n  Failed IDs: {failed[:10]}")
            if len(failed) > 10:
                print(f"  ... and {len(failed)-10} more")

        print(f"\n  Output: {OUTPUT_GPKG}")
        print(f"  Log:    {LOG_FILE}")
        print("=" * 80)
        print("DONE!")


if __name__ == "__main__":
    main()

REFERENCE AREA CREATION
Started: 2026-04-14 10:14:42

Loading data...
  ✓ NHDA:             839 clusters
  ✓ ATKIS:            511,977 polygons  |  columns: ['LAND', 'MODELLART', 'OBJART', 'OBJART_TXT', 'OBJID', 'BEGINN', 'ENDE', 'HDU_X', 'FDV_X', 'DLU', 'EDU', 'IWN', 'AGT', 'BEB', 'BEZ', 'FGT', 'FKT', 'LGT', 'NAM', 'PEG', 'ZNM', 'ZUS', 'geometry']
  ✓ New buildings:    1,811,517 footprints
  ✓ Existing bldgs:   8,271,554 footprints
  ✓ All buildings:    10,083,071 footprints
  ✓ ATKIS filtered to classes [41001, 41006]: 511,977 polygons
  ✓ CRS: EPSG:25832

Building spatial indices...
  ✓ Spatial indices ready

Creating ATKIS union...
    Chunk 1/1024 (500 polygons)...
    Chunk 2/1024 (500 polygons)...
    Chunk 3/1024 (500 polygons)...
    Chunk 4/1024 (500 polygons)...
    Chunk 5/1024 (500 polygons)...
    Chunk 6/1024 (500 polygons)...
    Chunk 7/1024 (500 polygons)...
    Chunk 8/1024 (500 polygons)...
    Chunk 9/1024 (500 polygons)...
    Chunk 10/1024 (500 polygons)...
    C